# Unidad 40. Distribuciones y calidad de una variable

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Una distribución resume cómo se reparten los valores de una variable. Antes de interpretarla, hay que saber qué mide la variable, cuántas observaciones hay y qué problemas de calidad contienen los datos.

## Resultados de aprendizaje

Al terminar podrás:

- revisar valores ausentes antes de representar una variable numérica;
- crear e interpretar histogramas, estimaciones KDE y funciones de distribución acumulada (ECDF);
- justificar una elección de intervalos y distinguir frecuencia, proporción y densidad;
- comparar distribuciones sin convertir una pauta visual en una afirmación causal.

**Prerequisitos:** unidad 39 y el manejo básico de `DataFrame` de Pandas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def ruta_datos(nombre):
    candidatas = [
        Path("datos") / "seaborn" / nombre,
        Path("..") / "datos" / "seaborn" / nombre,
    ]
    for candidata in candidatas:
        if candidata.exists():
            return candidata
    raise FileNotFoundError(
        f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso."
    )

mediciones = pd.read_csv(ruta_datos("40_mediciones_calidad_aire.csv"))
mediciones.head()

## 1. Antes del gráfico: qué observamos y qué falta

El archivo contiene lecturas **simuladas** de PM2.5 de tres zonas. Una observación no es una medición real ni permite hacer afirmaciones sobre una ciudad. La variable `pm25_ug_m3` es numérica y continua: puede tomar valores decimales.

Los valores ausentes no son ceros. Un cero indicaría una concentración medida igual a cero; un valor ausente significa que desconocemos la lectura. Para explorar esta variable eliminaremos únicamente las filas sin PM2.5, conservando el resto de columnas y documentando la decisión.

In [ ]:
mediciones.info()

ausentes = mediciones.isna().sum()
print(ausentes)

pm25_limpias = mediciones.dropna(subset=["pm25_ug_m3"]).copy()
print(f"Filas originales: {len(mediciones)}")
print(f"Filas para PM2.5: {len(pm25_limpias)}")
pm25_limpias.describe()

La eliminación anterior es razonable para un ejemplo breve porque solo faltan dos lecturas y estamos estudiando una única variable. En otro contexto podrías investigar por qué faltan datos, informar del porcentaje perdido o usar un método de imputación justificado. Nunca ocultes esa decisión.

## 2. Histograma: valores agrupados en intervalos

Un histograma divide el eje horizontal en **intervalos** o *bins* y cuenta cuántas observaciones caen en cada uno. Con pocos intervalos se pierde detalle; con demasiados, el ruido de una muestra pequeña puede parecer una pauta importante. No existe un número de intervalos universalmente correcto: pruébalo, explica tu elección y mira los datos originales.

Con `stat="count"`, la altura es el número de observaciones. No confundas ese conteo con una probabilidad ni con una densidad.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    bins=8,
    stat="count",
    color="#4477AA",
    edgecolor="white",
    ax=ax,
)
ax.set(
    title="Distribución simulada de PM2.5",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Número de lecturas",
)
plt.show()

### Comprueba que lo has entendido

Antes de modificar el código, predice qué ocurrirá si cambias `bins=8` por `bins=16`: ¿cambiará el número total de lecturas? ¿Cambiará el aspecto del gráfico? Ejecuta la variante y responde.

El total no cambia, porque las observaciones son las mismas. Sí cambia el aspecto: al haber intervalos más estrechos, algunos pueden contener muy pocas observaciones. Por eso una conclusión no debe depender de una única elección de bins.

## 3. KDE: una curva suavizada, no datos adicionales

Una estimación de densidad por núcleo o **KDE** dibuja una curva suave que resume dónde se concentran los valores. Es útil para ver una forma general, pero depende de un parámetro de suavizado y puede sugerir detalles que los datos no sostienen. No la interpretes como una frecuencia exacta ni como la distribución verdadera de una población.

En el gráfico siguiente se conserva el histograma con `stat="density"`. Ahora el área total de las barras y de la curva se interpreta como 1; la altura ya no representa un número de lecturas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    bins=8,
    stat="density",
    color="#AABBDD",
    edgecolor="white",
    ax=ax,
)
sns.kdeplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    color="#225588",
    linewidth=2,
    ax=ax,
)
ax.set(
    title="Histograma y KDE de PM2.5",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Densidad",
)
plt.show()

## 4. ECDF: proporción acumulada sin elegir intervalos

La **función de distribución acumulada empírica** (ECDF) muestra, para cada valor horizontal, qué proporción de observaciones es menor o igual que él. No exige elegir bins ni suavizado. La curva sube en escalones y termina en 1.

Por ejemplo, si la ECDF vale aproximadamente 0,50 en 18 µg/m³, alrededor de la mitad de las lecturas limpias no supera ese valor. No afirma nada sobre lecturas futuras ni sobre zonas no observadas.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.ecdfplot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    hue="zona",
    linewidth=2,
    ax=ax,
)
ax.set(
    title="Proporción acumulada de lecturas por zona",
    xlabel="PM2.5 (µg/m³)",
    ylabel="Proporción acumulada",
)
ax.legend(title="Zona")
plt.show()

Una curva más desplazada a la derecha indica valores generalmente mayores dentro de esa muestra. Para interpretar la diferencia necesitarías saber si los sensores son comparables, cuándo se tomaron las lecturas, cuántas faltan y qué otras variables intervienen.

## 5. Funciones de nivel de figura: `displot`

`sns.displot` crea una figura completa para distribuciones. Puede construir histogramas, KDE o ECDF y, más adelante, dividirlos en paneles. Como en `relplot`, el objeto resultante administra su propia figura. Úsalo cuando no necesitas insertarlo en unos ejes de Matplotlib ya existentes.

In [ ]:
grafico_distribucion = sns.displot(
    data=pm25_limpias,
    x="pm25_ug_m3",
    col="zona",
    kind="hist",
    bins=5,
    color="#4477AA",
    height=3.2,
    aspect=1,
)
grafico_distribucion.set_axis_labels("PM2.5 (µg/m³)", "Número de lecturas")
grafico_distribucion.figure.suptitle("Histogramas por zona simulada", y=1.05)
plt.show()

## Práctica guiada · comparar una decisión de intervalo

Construye dos histogramas de `pm25_ug_m3` en una misma figura, con `bins=5` y `bins=12`. Usa `sharey=True`, pon títulos claros y conserva el eje vertical como número de lecturas. Después escribe dos frases: una sobre una pauta que aparezca en ambos paneles y otra sobre un detalle que cambie al modificar los bins.

La comparación te ayuda a comprobar si una conclusión depende excesivamente de una elección de diseño.

In [ ]:
# Completa la práctica guiada aquí.
# Crea fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(10, 4)).


## Ejercicios

Usa siempre `pm25_limpias` para gráficos de PM2.5, salvo que el enunciado pida explícitamente analizar los valores ausentes.

### Ejercicio 40.01 · Informe mínimo de calidad

Crea una tabla con el número de filas, valores ausentes y porcentaje de valores ausentes de cada columna. Verifica con una aserción que `pm25_ug_m3` tiene dos valores ausentes. Explica por qué no sería correcto reemplazarlos automáticamente por cero.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.01.


### Ejercicio 40.02 · Histograma que se pueda leer

Crea un histograma de `temperatura_c` con seis intervalos. Elige un color, añade bordes claros, título y etiquetas con unidades. Indica en un comentario qué representa la altura de cada barra.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.02.


### Ejercicio 40.03 · Frecuencia frente a densidad

Representa PM2.5 con histograma y KDE en los mismos ejes. Configura el histograma con `stat="density"` y explica por qué el eje vertical ya no debe llamarse «Número de lecturas».

In [ ]:
# Escribe aquí tu solución del ejercicio 40.03.


### Ejercicio 40.04 · Pregunta acumulada

Con una ECDF por zona, responde: **«¿Qué zonas tienen todas sus lecturas limpias de PM2.5 por debajo de 20 µg/m³?»** Comprueba primero tu respuesta con una operación de Pandas. No extrapoles la respuesta fuera de este conjunto simulado.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.04.


### Ejercicio 40.05 · Miniproyecto: distribución por zona

Elabora una figura que permita comparar la distribución de PM2.5 entre las tres zonas simuladas. Puedes usar una ECDF con `hue`, un `displot` con paneles o una combinación razonada de histograma y KDE. Debe incluir unidades, una leyenda o títulos de panel y una breve justificación de tu elección.

Escribe tres conclusiones: una pauta visible, una limitación de los datos y una comprobación adicional que necesitarías antes de tomar una decisión real.

In [ ]:
# Escribe aquí tu solución del ejercicio 40.05.


## Autoevaluación y cierre

1. ¿Qué diferencia hay entre un valor ausente y un cero?
2. ¿Qué cambia y qué no cambia al modificar los bins de un histograma?
3. ¿Qué representa el eje vertical con `stat="density"`?
4. ¿Qué ventaja tiene una ECDF frente a un histograma?
5. ¿Por qué una diferencia entre zonas no prueba por sí sola que la zona cause los valores?

En la siguiente unidad pasarás de una variable a relaciones entre dos variables cuantitativas.